<a href="https://colab.research.google.com/github/Andrian0s/ML4NLP1-2026-Tutorial-Notebooks/blob/main/tutorials_notebooks_in_class_2026/W09_introduction_to_gliner_nuextract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Introduction to GLiNER**

In [1]:
%pip install -q transformers==5.16.1 gliner==0.2.29 "gliner2[local]==2.1.0" sentencepiece protobuf

import torch
# Pick the best available device: NVIDIA GPU, then Apple Silicon GPU, then CPU
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(device)

mps


## Model information

GLiNER (Generalist Lightweight Named Entity Recognizer) is a **modern, efficient model for Named Entity Recognition (NER)**.
Unlike traditional NER systems that require task-specific fine-tuning, GLiNER can **generalize across domains and entity types** with **minimal supervision**.

It’s particularly useful when you need:

* **Zero-shot NER**, without retraining large models.
* **Fast inference** with limited computational resources.
* **Custom entity extraction** from text (e.g., companies, genes, product names, etc.).

### How GLiNER Works

GLiNER reframes NER as a **text-text matching problem**:

* Entities and text are placed in the same prompt.
* Span embeddings are calculated by using up to 12 consecutive tokens: First and last token representations are concatenated and passed through a feed-forward network for refinement.
* Entity embeddings are calculated by taking the **first** token of each entity's token representations and passing it through a feed-forward network for refinement (different than the span's FFN).
* Entity and span embeddings are in the same latent space.
* Scores are calculated between all possible combinations of (entity embedding) <-> (span embedding)


More details in this [paper.](https://arxiv.org/abs/2311.08526)

## Trying out GLiNER

You can experiment with the model on this [HuggingFace space](https://huggingface.co/spaces/urchade/gliner_multiv2.1).

In [2]:
from gliner import GLiNER


model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1").to(device)

In [3]:
text = "Marie Curie discovered radium in Paris in 1898."

# Define entity types in natural language
entity_descriptions = [
  "person name",
  "chemical element",
  "city",
  "date",
  "id number"
]

# Run prediction
entities = model.predict_entities(
    text,
    entity_descriptions,
)
entities

[{'start': 0,
  'end': 11,
  'text': 'Marie Curie',
  'label': 'person name',
  'score': 0.976923406124115},
 {'start': 23,
  'end': 29,
  'text': 'radium',
  'label': 'chemical element',
  'score': 0.9522209167480469},
 {'start': 33,
  'end': 38,
  'text': 'Paris',
  'label': 'city',
  'score': 0.9851383566856384},
 {'start': 42,
  'end': 46,
  'text': '1898',
  'label': 'date',
  'score': 0.9774954915046692}]

You can process many texts at once with `.inference` and the `batch_size` argument. On a GPU, this makes inference noticeably faster.

In [4]:
texts = ["Marie Curie discovered radium in Paris in 1898."] * 10

# Define entity types in natural language
entity_descriptions = [
  "person name",
  "chemical element",
  "city",
  "date",
  "id number"
]

# Run prediction
entities = model.inference(
    texts,
    entity_descriptions,
    batch_size=5
)
entities

[[{'start': 0,
   'end': 11,
   'text': 'Marie Curie',
   'label': 'person name',
   'score': 0.9769232869148254},
  {'start': 23,
   'end': 29,
   'text': 'radium',
   'label': 'chemical element',
   'score': 0.9522200226783752},
  {'start': 33,
   'end': 38,
   'text': 'Paris',
   'label': 'city',
   'score': 0.9851382970809937},
  {'start': 42,
   'end': 46,
   'text': '1898',
   'label': 'date',
   'score': 0.9774957895278931}],
 [{'start': 0,
   'end': 11,
   'text': 'Marie Curie',
   'label': 'person name',
   'score': 0.9769232869148254},
  {'start': 23,
   'end': 29,
   'text': 'radium',
   'label': 'chemical element',
   'score': 0.9522200226783752},
  {'start': 33,
   'end': 38,
   'text': 'Paris',
   'label': 'city',
   'score': 0.9851382970809937},
  {'start': 42,
   'end': 46,
   'text': '1898',
   'label': 'date',
   'score': 0.9774957895278931}],
 [{'start': 0,
   'end': 11,
   'text': 'Marie Curie',
   'label': 'person name',
   'score': 0.9769232869148254},
  {'start':

## GLiNER2: one small model for several extraction tasks

[GLiNER2](https://arxiv.org/abs/2507.18546) (2025) extends the GLiNER idea beyond NER. A single encoder of a few hundred million parameters recognises entities, classifies the text, extracts relations and fills in structured records. All tasks are described in one schema and computed in one forward pass, and the model still runs on a CPU.

Below we use the multilingual checkpoint [`fastino/gliner2.5-multi-v1`](https://huggingface.co/fastino/gliner2.5-multi-v1) (287M parameters). Note that GLiNER2 comes with its own library, `gliner2`, which is different from the `gliner` library above.

In [5]:
from gliner2 import AutoExtractor

extractor = AutoExtractor.from_pretrained("fastino/gliner2.5-multi-v1", map_location=device)

nlp_history = (
    "Natural language processing has its roots in the 1950s. Already in 1950, Alan Turing published an article "
    "titled \"Computing Machinery and Intelligence\" which proposed what is now called the Turing test as a "
    "criterion of intelligence, though at the time that was not articulated as a problem separate from artificial "
    "intelligence. The proposed test includes a task that involves the automated interpretation and generation "
    "of natural language."
)

# One schema for three tasks: entities, a text label and a structured record
schema = (
    extractor.create_schema()
    .entities(["person", "scientific field", "test or method"])
    .classification("topic", ["history of science", "sports", "politics", "medicine"])
    .structure("publication", mode="natural", anchor="title")
        .field("title", dtype="str", cardinality="required_one")
        .field("author", dtype="str")
        .field("year", dtype="str")
)

extractor.extract(nlp_history, schema)

[transformers] You are using a model of type `gliner2` to instantiate a model of type `extractor`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


[transformers] You are using a model of type `gliner2` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


{'publication': [{'title': 'Computing Machinery and Intelligence',
   'author': 'Alan Turing',
   'year': None}],
 'entities': {'person': ['Alan Turing'],
  'scientific field': ['Natural language processing', 'intelligence'],
  'test or method': ['Turing test']},
 'topic': 'history of science'}

GLiNER2 only selects spans of the input text and links them through the schema. In our run, it found the title and the author of the publication but left the year empty, although "1950" appears in the same sentence. We will look at the same text with NuExtract below.

# **Introduction to NuExtract (NOTE: This is not included in the exercise)**

## Model information

**NuExtract3** (2026) is a 4B-parameter vision-language model for structured information extraction, fine-tuned by NuMind from Qwen3.5-4B.
Instead of focusing on fixed entity types (like traditional NER), it receives a JSON template and a document, and the model responds with the same JSON, but with the values filled in from the document.

It’s particularly useful when you need:

* **Schema-guided extraction**, not just label classification.
* **Documents of any kind**: plain text, but also scans, receipts, forms or tables given as images.
* **Multilingual extraction** across English, French, German, Spanish, and many more languages.
* **Typed outputs**: each field of the template declares what kind of value it expects, for example a verbatim string, a date or a number.

### How NuExtract Works

NuExtract reframes extraction as a **structured generation task** guided by a JSON template:

* Inputs combine a **document** (text, images or both) and a **JSON template**. The leaves of the template give the type of each field, for example `"verbatim-string"` (copied exactly from the document), `"string"` (may be paraphrased), `"integer"`, `"date"`, or a list of allowed values. Optional instructions and a few examples for the template can also be given.
* The model, a generative LLM, writes the filled-in JSON token by token. Fields that cannot be found in the document come back as `null` or `[]`.
* Besides extraction, it can convert a document into Markdown, and it has an optional reasoning ("thinking") mode for difficult documents.

More details on the model [here.](https://huggingface.co/numind/NuExtract3)


## Trying out NuExtract

You can experiment with the model on this [HuggingFace space](https://huggingface.co/spaces/numind/NuExtract3). Copy paste text from an article of your choice, define your custom JSON template and run inference.

### **Example:**

****Text:****

Natural language processing has its roots in the 1950s. Already in 1950, Alan Turing published an article titled "Computing Machinery and Intelligence" which proposed what is now called the Turing test as a criterion of intelligence, though at the time that was not articulated as a problem separate from artificial intelligence. The proposed test includes a task that involves the automated interpretation and generation of natural language.

****JSON template:****
```json
{
  "topic": "verbatim-string",
  "time_period": "string",
  "key_figures": [
    {
      "name": "verbatim-string",
      "contribution": "string"
    }
  ],
  "important_publications": [
    {
      "title": "verbatim-string",
      "year": "integer",
      "author": "verbatim-string"
    }
  ],
  "concepts_introduced": [
    {
      "name": "verbatim-string",
      "description": "string"
    }
  ],
  "related_fields": ["verbatim-string"]
}
```



****Model's output (non-thinking mode, greedy decoding):****
```json
{
  "topic": "Natural language processing",
  "time_period": "1950s",
  "key_figures": [
    {
      "name": "Alan Turing",
      "contribution": "Proposed the Turing test as a criterion of intelligence, which involves the automated interpretation and generation of natural language."
    }
  ],
  "important_publications": [
    {
      "title": "Computing Machinery and Intelligence",
      "year": 1950,
      "author": "Alan Turing"
    }
  ],
  "concepts_introduced": [
    {
      "name": "Turing test",
      "description": "A criterion of intelligence that involves a task requiring the automated interpretation and generation of natural language."
    }
  ],
  "related_fields": [
    "artificial intelligence"
  ]
}
```

Compared with GLiNER2 above, NuExtract also returned the year, as a number, and rephrased the contributions in its own words, since these fields are typed `"integer"` and `"string"`. In return, it is more than ten times larger (4B versus 287M parameters) and needs a GPU to run at a reasonable speed.